In [69]:

import sys
import itertools
from tqdm.auto import tqdm
import pathlib
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

import datasets
from contextlib import nullcontext
import torch
from torch import nn
from transformers import (
    Trainer,
    TrainingArguments,
    LlamaTokenizer,
    LlamaForSequenceClassification,
    TrainerCallback,
    default_data_collator,
)
from peft import (
    get_peft_model,
    LoraConfig,
    TaskType,
    prepare_model_for_int8_training,
)


sys.path.append("../src")
sys.path.append("../config")
from utils import number_split, create_mix


from process_HateSpeech import load_HateSpeech_dynGen, load_HateSpeech_wsf
from process_SHAC import load_process_SHAC



In [2]:
df_dynGen = load_HateSpeech_dynGen()
df_wsf = load_HateSpeech_wsf()

In [3]:
df_shac = load_process_SHAC(replaceNA="all")

In [35]:
print(df_dynGen['label_binary'].sum()/len(df_dynGen))
print(df_wsf['label_binary'].sum()/len(df_wsf))

0.5389607233132413
0.1117443707371765


In [43]:
## Hate Speech
df0 = df_dynGen
df1 = df_wsf
df_split_label = "label_binary"

p_pos_train_z0_ls = [0.3] # probability of training set examples drawn from site/domain z0 being positive
p_pos_train_z1_ls = [0.3] # probability of test set examples drawn from site/domain z1 being positive
p_mix_z1_ls     = np.arange(0.1, 0.9, 0.1) 
n_test = 1000


## SHAC

# df_split_label = "Drug"
# df_shac_uw = df_shac.query("location == 'uw'").reset_index(drop=True)
# df_shac_mimic = df_shac.query("location == 'mimic'").reset_index(drop=True)

# df0 = df_shac_uw
# df1 = df_shac_mimic
# p_pos_train_z0_ls = np.arange(0, 1, 0.1)
# p_pos_train_z1_ls = np.arange(0, 1, 0.1)
# p_mix_z1_ls = np.arange(0, 1, 0.05)

# n_test = 200

##### Split

train_test_ratio = 4


numvals = 1023
base = 1.1
alpha_test_ls = np.power(base, np.arange(numvals)) / np.power(base, numvals // 2)



valid_full_settings = []
for combination in itertools.product(
    p_pos_train_z0_ls, p_pos_train_z1_ls, p_mix_z1_ls, alpha_test_ls
):
    number_setting = number_split(
        p_pos_train_z0=combination[0],
        p_pos_train_z1=combination[1],
        p_mix_z1=combination[2],
        alpha_test=combination[3],
        train_test_ratio=train_test_ratio,
        n_test=n_test,
        verbose=False,
    )

    
            
    if number_setting is not None:
        if np.all([number_setting[k] >= 10 for k in list(number_setting.keys())[:-1]]):
            valid_full_settings.append(number_setting)





valid_n_full_settings = []

for c in tqdm(valid_full_settings):
        c = c.copy()
        # create train/test split according to stats
        dfs = create_mix(df0=df0, df1=df1, target=df_split_label, setting=c, sample=False, 
                         seed=222
                        )

        if dfs is None:
            continue
        
        valid_n_full_settings.append(c)

  0%|          | 0/468 [00:00<?, ?it/s]

/home/NETID/xiruod/projects/DeconDTN/notebooks_xiruo/../src/utils.py:263: UserWarning: Set sample equals to True or augment current dataset.
  warnings.warn("Set sample equals to True or augment current dataset.")


In [44]:
tmp = [x['mix_param_dict'] for x in valid_n_full_settings]

In [45]:
df = pd.DataFrame(tmp)

In [48]:
df.query("(alpha_train == 1) and (C_z == 0.5) and (alpha_test==1)")

,p_pos_train_z0,p_pos_train_z1,p_pos_train,p_pos_test,p_mix_z0,p_mix_z1,alpha_train,alpha_test,p_pos_test_z0,p_pos_test_z1,C_y,C_z
247,0.3,0.3,0.3,0.3,0.5,0.5,1.0,1.0,0.3,0.3,0.3,0.5


In [ ]:
247

In [74]:
valid_n_full_settings[247]

{'n_train': 4000,
 'n_test': 1000,
 'n_z0_pos_train': 600,
 'n_z0_neg_train': 1400,
 'n_z0_pos_test': 150,
 'n_z0_neg_test': 350,
 'n_z1_pos_train': 600,
 'n_z1_neg_train': 1400,
 'n_z1_pos_test': 150,
 'n_z1_neg_test': 350,
 'mix_param_dict': {'p_pos_train_z0': 0.3,
  'p_pos_train_z1': 0.3,
  'p_pos_train': 0.3,
  'p_pos_test': 0.3,
  'p_mix_z0': 0.5,
  'p_mix_z1': 0.5,
  'alpha_train': 1.0,
  'alpha_test': 1.0,
  'p_pos_test_z0': 0.3,
  'p_pos_test_z1': 0.3,
  'C_y': 0.3,
  'C_z': 0.5}}

In [72]:
from sampling_numbers import *

In [73]:
HateSpeech_DICT['PickC-0']

{'PickC-0': {'p_pos_train_z0_ls': [0.3],
  'p_pos_train_z1_ls': [0.3],
  'p_mix_z1_ls': array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8])}}